# Experiment: Case 001 Immune Transfusion Record Gate

Objective: rank the missing immune/transfusion records that could explain weekly transfusion or poor red-cell survival in case-001.

Success criteria:
- do not diagnose the reported autoimmune issue;
- separate disease-severity questions from transfusion-medicine questions;
- produce a compact doctor-facing record request;
- keep every output de-identified and public-repo safe.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass


@dataclass(frozen=True)
class RecordGate:
    """One missing record family for immune/transfusion interpretation."""

    name: str
    explains_weekly_interval: bool
    changes_safety_interpretation: bool
    blocks_trial_screening: bool
    source_backing: int
    exact_request: str


def priority_score(gate: RecordGate) -> int:
    """Score records by usefulness for clinician routing."""
    score = gate.source_backing
    if gate.explains_weekly_interval:
        score += 3
    if gate.changes_safety_interpretation:
        score += 3
    if gate.blocks_trial_screening:
        score += 2
    return score

## Plan

Hypothesis: weekly transfusion cannot be interpreted from interval alone. The most useful next records are the ones that can distinguish disease severity from immune red-cell loss, delayed hemolysis, crossmatch difficulty, hypersplenism, and low-volume scheduling.


In [ ]:
gates = [
    RecordGate(
        name="Antibody screen and antibody identification",
        explains_weekly_interval=True,
        changes_safety_interpretation=True,
        blocks_trial_screening=True,
        source_backing=5,
        exact_request=("antibody screen history, named alloantibodies, and dates"),
    ),
    RecordGate(
        name="DAT/direct Coombs with IgG and C3 specificity",
        explains_weekly_interval=True,
        changes_safety_interpretation=True,
        blocks_trial_screening=True,
        source_backing=5,
        exact_request="DAT result, method, IgG/C3 pattern, and date",
    ),
    RecordGate(
        name="Rh C/c, D, E/e, Kell matching policy",
        explains_weekly_interval=False,
        changes_safety_interpretation=True,
        blocks_trial_screening=True,
        source_backing=5,
        exact_request="blood-bank matching policy and antigen profile",
    ),
    RecordGate(
        name="Extended red-cell phenotype or genotype",
        explains_weekly_interval=False,
        changes_safety_interpretation=True,
        blocks_trial_screening=True,
        source_backing=4,
        exact_request=(
            "red-cell antigen phenotype or genotype, especially if serology "
            "is confounded by recent transfusion or antibodies"
        ),
    ),
    RecordGate(
        name="Transfusion dates, volume, and pre/post Hb",
        explains_weekly_interval=True,
        changes_safety_interpretation=False,
        blocks_trial_screening=True,
        source_backing=5,
        exact_request=(
            "last 6-12 months of dates, units or ml, body weight, "
            "pre-transfusion Hb, and post-transfusion Hb when available"
        ),
    ),
    RecordGate(
        name="Delayed hemolytic reaction history",
        explains_weekly_interval=True,
        changes_safety_interpretation=True,
        blocks_trial_screening=False,
        source_backing=4,
        exact_request=(
            "jaundice, dark urine, fever, back pain, malaise, or Hb fall "
            "5-14 days after transfusion"
        ),
    ),
    RecordGate(
        name="Spleen status and hypersplenism markers",
        explains_weekly_interval=True,
        changes_safety_interpretation=False,
        blocks_trial_screening=True,
        source_backing=4,
        exact_request="spleen size/status, platelet trend, and WBC trend",
    ),
]

ranked = sorted(gates, key=priority_score, reverse=True)
[(gate.name, priority_score(gate)) for gate in ranked]

## Result

The highest-value records are antibody history, DAT/direct Coombs specificity, matching policy, and transfusion-volume/Hb trend. These records can change the interpretation of weekly transfusion without changing any treatment in this repo.


In [ ]:
doctor_packet = {
    "current_label": "immune_transfusion_packet_missing",
    "top_requests": [gate.exact_request for gate in ranked[:5]],
    "interpretation_boundary": (
        "weekly transfusion plus reported autoimmune issue requires "
        "blood-bank records before candidate relevance can be inferred"
    ),
    "research_consequence": (
        "HbF and globin-balance candidates must also pass mature "
        "red-cell hemolysis and immune/transfusion safety gates"
    ),
}

doctor_packet

## Durable Conclusion

Case-001 should carry `immune_transfusion_packet_missing` until antibody screen, DAT/direct Coombs specificity, matching policy, transfusion volume, and red-cell survival clues are available. This does not diagnose AIHA or alloimmunization; it only prevents the repo from misreading weekly transfusion as genotype-only disease severity.
